In [ ]:
import numpy as np
from dataclasses import replace
import matplotlib.pyplot as plt

from pathtracer import Renderer, display_encode

In [ ]:
renderer = Renderer("scenes/cornell.json")

In [ ]:
cam = replace(renderer.default_camera, position=(0.0, 0.0, 6.0), yaw_degrees=0.0, pitch_degrees=0.0)
frame = renderer.render(camera=cam, aovs=("luminance",), width=64, height=32, samples=16, show_sky=False)

In [ ]:
x = frame["luminance"]
print(f"Min: {x.min()}, Max: {x.max()}")

plt.imshow(x)
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
from scipy.signal import find_peaks

fig = plt.figure(figsize=(12, 4), constrained_layout=True)
gs = fig.add_gridspec(1, 3)

profile = x.sum(axis=(0, 2)) if x.ndim == 3 else x.sum(axis=0)
top_k = 7
all_peaks, props = find_peaks(profile, prominence=0)
top_idx = np.argsort(props["prominences"])[-top_k:]
peaks = all_peaks[top_idx]

ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(profile, color="#4a7fb5", lw=1.0)
ax0.plot(peaks, profile[peaks], "x", color="#d1615d", ms=6, mew=1.5)
ax0.set_title("Column Profile")
ax0.set_xlabel("x (pixels)"), ax0.set_ylabel("summed luminance")
ax0.grid(alpha=0.25)
ax0.spines[["top", "right"]].set_visible(False)

row_profile = x.sum(axis=(1, 2)) if x.ndim == 3 else x.sum(axis=1)
ax = fig.add_subplot(gs[0, 1])
ax.plot(row_profile, color="#d1615d", lw=1.0)
ax.set_title("Row Profile")
ax.set_xlabel("y (pixels)")
ax.set_ylabel("summed luminance")
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

ax = fig.add_subplot(gs[0, 2])
ax.hist(x.ravel(), bins=40, color="#5a9e6f", edgecolor="white", linewidth=0.3)
ax.set_yscale("log")
ax.set_title("Magnitude Distribution")
ax.set_xlabel("luminance per pixel")
ax.set_ylabel("pixel count")
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)